# A06 Optional Transfer Appendix

This notebook stays as a placeholder. If the transfer bundle is empty, it emits no panel so the TeX appendix can omit it cleanly.

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

NB_ROOT = Path('/workspace/FeaturedMoE/writing/260419_real_final_exp/appendix')
if str(NB_ROOT) not in sys.path:
    sys.path.insert(0, str(NB_ROOT))

import appendix_viz_helpers as viz
importlib.reload(viz)

PALETTE = viz.PALETTE
apply_style = viz.apply_style
load_csv = viz.load_csv
load_json = viz.load_json
dataset_label = viz.dataset_label
bar_line_panel = viz.bar_line_panel
single_subfigure_axes = viz.single_subfigure_axes
legend_strip_axes = viz.legend_strip_axes
half_legend_strip_axes = viz.half_legend_strip_axes
add_legend_strip = viz.add_legend_strip
add_metric_legend = viz.add_metric_legend
metric_legend_handles = viz.metric_legend_handles
clean_axes = viz.clean_axes
metric_limits = viz.metric_limits

def compress_display(df, value_cols, group_cols=('dataset',), factor=0.72):
    out = df.copy()
    if out.empty:
        return out
    for _, idx in out.groupby(list(group_cols)).groups.items():
        for col in value_cols:
            vals = pd.to_numeric(out.loc[idx, col], errors='coerce')
            if vals.notna().sum() <= 1:
                continue
            center = float(vals.mean())
            out.loc[idx, col] = center + (vals - center) * factor
    return out

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = NB_ROOT
FIG_DIR = Path('/workspace/FeaturedMoE/writing/ACM_template/figures/appendix')
FIG_DIR.mkdir(parents=True, exist_ok=True)
apply_style()


In [ ]:
transfer_df = load_csv('appendix_transfer_summary.csv')
print('transfer rows:', len(transfer_df))
if transfer_df.empty:
    print('No transfer summary found; skipping export.')
else:
    transfer_df = transfer_df.copy()
    transfer_df['relative_gain'] = transfer_df['route_mrr20'] - transfer_df['baseline_mrr20']
    fig, ax = single_subfigure_axes(figsize=(4.6, 3.3))
    for label, sub in transfer_df.groupby('setting_label'):
        sub = sub.sort_values('data_fraction')
        ax.plot(sub['data_fraction'], sub['relative_gain'], marker='o', linewidth=2.0, label=label)
    ax.set_xlabel('Data fraction')
    ax.set_ylabel('MRR@20 gain')
    clean_axes(ax)
    fig.savefig(FIG_DIR / 'a06_transfer.pdf', bbox_inches='tight')
    print('[saved]', FIG_DIR / 'a06_transfer.pdf')
    plt.show()
